# 02 — Modeling Pipeline (Baseline)

## 01 - Imports


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import seaborn as sns
sns.set_theme(palette="bright", style="white")

from pathlib import Path

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge, ElasticNet

from sklearn.model_selection import train_test_split, GridSearchCV, KFold, cross_val_score
from sklearn.metrics import root_mean_squared_error


## 02 — Global Config


In [2]:
RANDOM_STATE = 42
TEST_SIZE = 0.2
N_SPLITS = 5
TARGET = "median_house_value"

SCORING = "neg_root_mean_squared_error"

## 03 — Load Data


In [3]:
DATA_PATH = Path("../data/housing.csv")

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
df.head()


Shape: (20640, 10)


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


## 04 — Feature engineering


In [4]:
def add_features(df):
    df_new = df.copy()

    households = df_new["households"].replace(0, np.nan)
    total_rooms = df_new["total_rooms"].replace(0, np.nan)

    df_new["population_per_households"] = df_new["population"] / households
    df_new["total_rooms_per_households"] = df_new["total_rooms"] / households
    df_new["total_bedrooms_per_households"] = df_new["total_bedrooms"] / households
    df_new["total_bedrooms_per_total_rooms"] = df_new["total_bedrooms"] / total_rooms

    return df_new

df_fe = add_features(df)

print(f"shape dataframe antigo: {df.shape}")
print(f"shape dataframe novo: {df_fe.shape}")

df_fe[
    [
        "population_per_households",
        "total_rooms_per_households",
        "total_bedrooms_per_households",
        "total_bedrooms_per_total_rooms",
    ]
].head()


shape dataframe antigo: (20640, 10)
shape dataframe novo: (20640, 14)


,population_per_households,total_rooms_per_households,total_bedrooms_per_households,total_bedrooms_per_total_rooms
0,2.555556,6.984127,1.023810,0.146591
1,2.109842,6.238137,0.971880,0.155797
2,2.802260,8.288136,1.073446,0.129516
3,2.547945,5.817352,1.073059,0.184458
4,2.181467,6.281853,1.081081,0.172096


## 05 — Split X e y


In [5]:
X = df_fe.drop(columns=[TARGET])
y = df_fe[TARGET]

display(X.head())
display(y.head())


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,ocean_proximity,population_per_households,total_rooms_per_households,total_bedrooms_per_households,total_bedrooms_per_total_rooms
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,NEAR BAY,2.555556,6.984127,1.023810,0.146591
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,NEAR BAY,2.109842,6.238137,0.971880,0.155797
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,NEAR BAY,2.802260,8.288136,1.073446,0.129516
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,NEAR BAY,2.547945,5.817352,1.073059,0.184458
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,NEAR BAY,2.181467,6.281853,1.081081,0.172096


0    452600.0
1    358500.0
2    352100.0
3    341300.0
4    342200.0
Name: median_house_value, dtype: float64

## 06 — Split and Stritify income

In [6]:
income_bins = [0.0, 1.5, 3.0, 4.5, 6.0, np.inf]
income_labels = [1, 2, 3, 4, 5]

income_strata = pd.cut(
    df_fe["median_income"],
    bins=income_bins,
    labels=income_labels,
    include_lowest=True,
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=income_strata,
)

print("Treino:", X_train.shape, y_train.shape)
print("Teste :", X_test.shape, y_test.shape)


Treino: (16512, 13) (16512,)
Teste : (4128, 13) (4128,)


## 07 — Defining Columns


In [7]:
cat_col = ["ocean_proximity"]
num_col = [c for c in X_train.columns if c not in cat_col]

print("Num cols:", len(num_col))
print("Cat cols:", len(cat_col))

Num cols: 12
Cat cols: 1


## BLOCO 08 — Pipelines de pré-processamento (numérico e categórico)


In [8]:
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, num_col),
        ("cat", categorical_pipeline, cat_col),
    ],
    remainder="drop",
)


## BLOCO 09 — Aux functions

In [12]:
def cv_rmse(pipeline, X_train, y_train):
    cv = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    scores = cross_val_score(
        pipeline,
        X_train,
        y_train,
        scoring=SCORING,
        cv=cv,
        n_jobs=-1,
    )
    rmse = -scores
    return rmse.mean(), rmse.std()

def evaluate_model(
    name,
    estimator,
    *,
    param_grid = None,
    do_search = False,
):
    base_pipeline = Pipeline(
        steps=[
            ("preprocess", preprocessor),
            ("model", estimator),
        ]
    )

    if do_search:
        cv = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
        search = GridSearchCV(
            base_pipeline,
            param_grid=param_grid,
            scoring=SCORING,
            cv=cv,
            n_jobs=-1,
        )
        search.fit(X_train, y_train)
        best_pipeline = search.best_estimator_
        best_params = search.best_params_
    else:
        best_pipeline = base_pipeline
        best_pipeline.fit(X_train, y_train)
        best_params = None

    cv_mean, cv_std = cv_rmse(best_pipeline, X_train, y_train)

    y_pred = best_pipeline.predict(X_test)
    test_rmse = root_mean_squared_error(y_test, y_pred)

    return {
        "model": name,
        "cv_rmse_mean": cv_mean,
        "cv_rmse_std": cv_std,
        "test_rmse": test_rmse,
        "best_params": best_params,
        "fitted_pipeline": best_pipeline,
    }

## BLOCO 10 — Baselines (Dummy, Linear, ElasticNet and Ridge)


In [13]:
results = []

# 1) Dummy
results.append(
    evaluate_model(
        "DummyRegressor(median)",
        DummyRegressor(strategy="median"),
    )
)

# 2) Linear Regression
results.append(
    evaluate_model(
        "LinearRegression",
        LinearRegression(),
    )
)

# 3) Ridge
ridge_grid = {
    "model__alpha": [0.01, 0.1, 1.0, 10.0, 100.0]
}
results.append(
    evaluate_model(
        "Ridge (GridSearch)",
        Ridge(random_state=RANDOM_STATE),
        param_grid=ridge_grid,
        do_search=True,
    )
)

# 4) Elastic Net
enet_grid = {
    "model__alpha": [0.01, 0.1, 1.0, 10.0],
    "model__l1_ratio": [0.1, 0.5, 0.9],
}
results.append(
    evaluate_model(
        "ElasticNet (GridSearch)",
        ElasticNet(random_state=RANDOM_STATE, max_iter=10000),
        param_grid=enet_grid,
        do_search=True,
    )
)

results_df = pd.DataFrame(
    [
        {k: v for k, v in r.items() if k not in ["fitted_pipeline"]}
        for r in results
    ]
).sort_values(by="cv_rmse_mean")

display(results_df)


,model,cv_rmse_mean,cv_rmse_std,test_rmse,best_params
3,ElasticNet (GridSearch),68351.566847,1051.897057,67006.236336,"{'model__alpha': 0.01, 'model__l1_ratio': 0.5}"
2,Ridge (GridSearch),68376.419474,1185.747955,66981.074155,{'model__alpha': 10.0}
1,LinearRegression,68399.119724,1228.644149,66894.977972,None
0,DummyRegressor(median),118936.974046,1327.613169,117256.683158,None


## 11 — Selecting best Model


In [14]:
best_row = results_df.iloc[0]
print("Melhor por CV:", best_row["model"])
print("CV RMSE:", best_row["cv_rmse_mean"], "±", best_row["cv_rmse_std"])
print("Test RMSE:", best_row["test_rmse"])

# Mostrar parâmetros quando houver GridSearch
if best_row["best_params"] is not None:
    print("\nBest params:")
    print(best_row["best_params"])


Melhor por CV: ElasticNet (GridSearch)
CV RMSE: 68351.56684684962 ± 1051.8970572333437
Test RMSE: 67006.23633571653

Best params:
{'model__alpha': 0.01, 'model__l1_ratio': 0.5}


## BLOCO 12 — Próximos passos (sem executar aqui)

- Se **Ridge/ElasticNet** vencerem com folga o Linear, mantenha regularização como padrão.
- Se a diferença entre CV e teste estiver grande:
  - checar leakage (pipeline fora do split),
  - checar outliers,
  - considerar transformação do target (log1p) em notebook separado (03_target_transform.ipynb).
